# Notebook 03 — Model 3: DistilBERT NER
**Technology**: `distilbert-base-uncased` — Lightweight Transformer (HuggingFace)  
**Environment**: Google Colab T4 GPU

---
### Setup Instructions
1. Upload these files to Colab:
   - `data/train.json`
   - `data/test.json`
   - `data/labels.json`
2. Set Runtime → **T4 GPU** before running
3. Run all cells top-to-bottom


In [1]:
# ── Install dependencies ─────────────────────────────────────────
!pip install transformers datasets seqeval accelerate -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [1]:
import json, os, random
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset
import evaluate

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# -- GPU check --
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  ⚠️  No GPU detected — training will be very slow. Enable T4 in Runtime settings.")


Device: cuda
  GPU : Tesla T4
  VRAM: 15.6 GB


In [2]:
# -- Load data --
# If files are in /content/, adjust path accordingly
DATA_DIR = Path("data")      # or Path("/content/data")

train_raw     = json.load(open(DATA_DIR / "train.json", encoding="utf-8"))
test_raw      = json.load(open(DATA_DIR / "test.json",  encoding="utf-8"))
ENTITY_LABELS = json.load(open(DATA_DIR / "labels.json"))

print(f"Train: {len(train_raw)} | Test: {len(test_raw)}")
print("Entity labels:", ENTITY_LABELS)


Train: 160 | Test: 40
Entity labels: ['College Name', 'Companies worked at', 'Degree', 'Designation', 'Email Address', 'Graduation Year', 'Location', 'Name', 'Skills', 'Years of Experience']


## 1. BIO Label Schema

In [3]:
# -- Build BIO label list --
# BIO = Begin / Inside / Outside tagging scheme
bio_labels = ["O"]
for label in ENTITY_LABELS:
    bio_labels.append(f"B-{label}")
    bio_labels.append(f"I-{label}")

label2id = {l: i for i, l in enumerate(bio_labels)}
id2label = {i: l for i, l in enumerate(bio_labels)}

print(f"Total BIO labels: {len(bio_labels)}")
for lbl in bio_labels[:7]:
    print(f"  {label2id[lbl]:3d} → {lbl}")
print("  ...")


Total BIO labels: 21
    0 → O
    1 → B-College Name
    2 → I-College Name
    3 → B-Companies worked at
    4 → I-Companies worked at
    5 → B-Degree
    6 → I-Degree
  ...


## 2. Tokenization & Label Alignment

In [4]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def align_labels(text, entities, tokenizer, label2id, max_length=512):
    """
    Convert character-level entity spans to token-level BIO labels.
    Uses character-level intermediate array → handles wordpiece correctly.
    """
    # Step 1: Build character-level label map
    char_label = {}
    for s, e, lbl in sorted(entities, key=lambda x: -(x[1]-x[0])):  # longest first
        for i in range(int(s), int(e)):
            if i not in char_label:
                char_label[i] = f"B-{lbl}" if i == int(s) else f"I-{lbl}"

    # Step 2: Tokenize
    encoding = tokenizer(text, truncation=True, max_length=max_length,
                         return_offsets_mapping=True, padding=False)
    offsets  = encoding.pop("offset_mapping")

    # Step 3: Map char labels → token labels
    token_labels = []
    for tok_s, tok_e in offsets:
        if tok_s == tok_e:               # Special token (CLS, SEP, PAD)
            token_labels.append(-100)
        else:
            raw_lbl = char_label.get(tok_s, "O")
            token_labels.append(label2id.get(raw_lbl, label2id["O"]))

    encoding["labels"] = token_labels
    return encoding

def build_dataset(raw_data, tokenizer, label2id):
    rows = []
    for text, ann in raw_data:
        enc = align_labels(text, ann["entities"], tokenizer, label2id)
        rows.append({k: v for k, v in enc.items()})
    return Dataset.from_list(rows)

print("Building HuggingFace datasets...")
train_dataset = build_dataset(train_raw, tokenizer, label2id)
test_dataset  = build_dataset(test_raw,  tokenizer, label2id)
print(f"✅ Train: {len(train_dataset)} | Test: {len(test_dataset)}")
print(f"   Features: {list(train_dataset.features.keys())}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Building HuggingFace datasets...
✅ Train: 160 | Test: 40
   Features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']


## 3. Load Model & Configure Training

In [5]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(bio_labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model: {MODEL_CHECKPOINT}")
print(f"   Total params    : {total_params:,}")
print(f"   Trainable params: {trainable:,}")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model: distilbert-base-uncased
   Total params    : 66,379,029
   Trainable params: 66,379,029


In [6]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)

    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        p_seq, l_seq = [], []
        for p_id, l_id in zip(pred_seq, label_seq):
            if l_id != -100:
                p_seq.append(id2label[p_id])
                l_seq.append(id2label[l_id])
        true_preds.append(p_seq)
        true_labels.append(l_seq)

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": round(results["overall_precision"], 4),
        "recall":    round(results["overall_recall"],    4),
        "f1":        round(results["overall_f1"],        4),
    }


In [7]:
training_args = TrainingArguments(
    output_dir          = "models/distilbert-ner",
    num_train_epochs    = 18,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate       = 2e-5,
    weight_decay        = 0.01,
    warmup_steps        = 0.2,
    eval_strategy = "epoch",
    save_strategy       = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model  = "f1",
    logging_dir         = "logs/distilbert",
    logging_steps       = 20,
    fp16                = torch.cuda.is_available(),
    report_to           = "none",
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = test_dataset,
    processing_class= tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
)
print("✅ Trainer ready")


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Trainer ready


## 4. Train

In [8]:
print(f"🚀 Starting DistilBERT fine-tuning...")
print(f"   Epochs: {training_args.num_train_epochs} | Batch: {training_args.per_device_train_batch_size}")

train_result = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Training time: {train_result.metrics['train_runtime']:.1f}s")
print(f"   Samples/sec  : {train_result.metrics['train_samples_per_second']:.1f}")


🚀 Starting DistilBERT fine-tuning...
   Epochs: 18 | Batch: 16


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,2.885878,0.000200,0.004700,0.000300
2,2.778762,1.669937,0.000000,0.000000,0.000000
3,2.778762,0.884925,0.000000,0.000000,0.000000
4,1.011340,0.738679,0.000000,0.000000,0.000000
5,1.011340,0.644863,0.000000,0.000000,0.000000
6,0.683446,0.587179,0.000000,0.000000,0.000000
7,0.683446,0.509940,0.091500,0.030200,0.045500
8,0.526595,0.459486,0.080600,0.034900,0.048700
9,0.526595,0.427780,0.060300,0.027900,0.038200
10,0.423213,0.419959,0.066200,0.041900,0.051300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ Training complete!
   Training time: 476.8s
   Samples/sec  : 6.0


## 5. Evaluate & Save Results

In [9]:
# -- Detailed per-entity evaluation --
predictions, labels_out, _ = trainer.predict(test_dataset)
preds = np.argmax(predictions, axis=2)

true_preds, true_labels = [], []
for pred_seq, label_seq in zip(preds, labels_out):
    p_seq, l_seq = [], []
    for p_id, l_id in zip(pred_seq, label_seq):
        if l_id != -100:
            p_seq.append(id2label[p_id])
            l_seq.append(id2label[l_id])
    true_preds.append(p_seq)
    true_labels.append(l_seq)

detailed = seqeval.compute(predictions=true_preds, references=true_labels)

print(f"{'Label':30s} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>6}")
print("-" * 55)
for ent_lbl in ENTITY_LABELS:
    if ent_lbl in detailed:
        m = detailed[ent_lbl]
        print(f"{ent_lbl:30s} {m['precision']:6.3f} {m['recall']:6.3f} {m['f1']:6.3f} {m['number']:6d}")
print("-" * 55)
print(f"{'OVERALL (micro)':30s} {detailed['overall_precision']:6.3f} "
      f"{detailed['overall_recall']:6.3f} {detailed['overall_f1']:6.3f}")


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Label                               P      R     F1    Sup
-------------------------------------------------------
College Name                    0.027  0.037  0.031     27
Companies worked at             0.000  0.000  0.000     94
Degree                          0.037  0.053  0.043     19
Designation                     0.226  0.211  0.218     90
Email Address                   0.425  0.721  0.534     43
Graduation Year                 0.000  0.000  0.000     16
Location                        0.000  0.000  0.000     66
Name                            0.477  0.775  0.590     40
Skills                          0.019  0.040  0.025     25
Years of Experience             0.000  0.000  0.000     10
-------------------------------------------------------
OVERALL (micro)                 0.233  0.195  0.212


In [11]:
# -- Save results --
Path("results").mkdir(exist_ok=True)

per_entity = {}
for ent_lbl in ENTITY_LABELS:
    if ent_lbl in detailed:
        m = detailed[ent_lbl]
        per_entity[ent_lbl] = {
            "precision": round(m['precision'], 4),
            "recall":    round(m['recall'], 4),
            "f1":        round(m['f1'], 4)
        }
    else:
        per_entity[ent_lbl] = {"precision": 0, "recall": 0, "f1": 0}

output = {
    "model": "Model 3 - DistilBERT",
    "overall": {
        "precision": round(detailed['overall_precision'], 4),
        "recall":    round(detailed['overall_recall'],    4),
        "f1":        round(detailed['overall_f1'],        4),
    },
    "per_entity": per_entity
}
json.dump(output, open("results/model3_results.json", "w"), indent=2)

# Save model
trainer.save_model("models/distilbert-ner-final")
print("✅ Results → results/model3_results.json")
print("✅ Model   → models/distilbert-ner-final/")
print(f"\n📌 DistilBERT Overall F1: {detailed['overall_f1']:.4f}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Results → results/model3_results.json
✅ Model   → models/distilbert-ner-final/

📌 DistilBERT Overall F1: 0.2124
